# W10 Lab — Agentic RAG over the EU AI Act

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week10/W10_lab_agentic_rag.ipynb)

**Goal.** A RAG program that always searches once before it answers, with the question as the search text, is **2-step RAG**: retrieval always comes before generation, and the program decides it. In **agentic RAG**, the search is a tool of an agent, and the model decides when and how to search: whether a message needs a search at all, what to search for, and whether to search again after reading a result.

| Part | What you learn |
|---|---|
| 1. The search tool | build the retriever over the Act and turn it into a tool |
| 2. The agent | build the agent; the model writes the query |
| 3. When to search | the agent answers without a search when none is needed |
| 4. Searching again | the agent searches again when a result is not enough |
| 5. A limit on the loop | stop a run after a fixed number of steps |
| 6. Memory | continue a conversation in a thread |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TYPE`: write the code between the `FILL IN` lines. The comments in the block and the *Do:* line before it say what to write.
- `# ▶ RUN · check`: just run; it records whether your code works.

**Submission.** This lab is collected and graded. It is the program of the Week 10 practice with five blocks for you to write: your name and student ID (Part 0), and 1.2, 2.1, 3.1, and 6.1. Run the notebook from top to bottom. The last cell prints PASS or FAIL for each check, and LAB COMPLETE when all of them pass. Then download the notebook with its outputs (**File → Download → Download .ipynb**) and upload it to the LMS. Due date: see the LMS.

It takes about 40 minutes. Do the Week 10 practice first. The text of the Act comes from EUR-Lex, without the recitals and the footnotes; only the version in the Official Journal of the EU is legally binding.

## 0. Setup

*Do:* run the four cells. Paste your API key into the second cell, and write your name and student ID into the third. The fourth cell makes the model object and downloads the text of the Act.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14" "langchain-text-splitters==1.1.3"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
### FILL IN (START) ###
STUDENT_NAME = ""   # your name
STUDENT_ID = ""     # your student ID
### FILL IN (END) ###
done = {"1.2": False, "2.1": False, "3.1": False, "6.1": False}   # the check cells record your results here
print("submitting as:", STUDENT_NAME, STUDENT_ID)

In [ ]:
# ▶ RUN · setup: just run
import urllib.request
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"
EMBEDDING_MODEL = "text-embedding-3-small"
model = init_chat_model(MODEL)     # one model object for the whole notebook

ACT_PATH = "eu_ai_act.md"
if not os.path.exists(ACT_PATH):   # download the text once
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/eu_ai_act.md", ACT_PATH)
act_text = open(ACT_PATH, encoding="utf-8").read()
print(len(act_text.split()), "words")

## 1. The Search Tool

An agent can search only through a tool. This part makes the retriever of Week 9 again and turns it into a tool.

### 1.1 The retriever

The cell repeats the retrieval part of the Week 9 RAG program: the same chunks, the same embedding model and vector store, and the same retriever, which returns the three most similar chunks for a query.

*Do:* run the cell. The embeddings take a few seconds.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "chapter"), ("##", "section")])
sections = []
for piece in header_splitter.split_text(act_text):
    if "section" in piece.metadata:                # one Document for each article or annex
        sections.append(piece)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
chunks = size_splitter.split_documents(sections)   # long sections become several chunks

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore(embedding_model)
ids = vector_store.add_documents(chunks)           # embeds every chunk once
retriever = vector_store.as_retriever(search_kwargs={"k": 3})   # the three nearest chunks for a query
print(len(sections), "sections,", len(chunks), "chunks,", len(ids), "in the store")

### 1.2 ✍️ The search tool

As in Practice 1, `@tool` turns a Python function into a tool, and the model sees only its name and its docstring. Here the function takes a query, runs the retriever of 1.1, and returns the three passages as one text, each under its article or annex label. The model writes the query itself, so the docstring says what a good query looks like.

*Do:* complete the line `docs = ...`: run `retriever` for the query. The lines after it write each chunk under its label and join them into one text. Then run the cell. It prints the name, description, and arguments of the tool and runs it once by hand.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
from langchain.tools import tool

@tool
def search_act(query: str) -> str:
    """Search the EU AI Act and return the three most relevant passages, each under its article or annex label.

    Args:
        query: What to search for, in a few words, for example "high-risk AI systems in employment".
    """
    ### FILL IN (START) ###
    docs = []                                      # replace: run the retriever for the query
    ### FILL IN (END) ###
    parts = []
    for doc in docs:
        parts.append("[" + doc.metadata["section"] + "]\n" + doc.page_content)   # each chunk under its label, e.g. "[Article 99: Penalties]"
    return "\n\n".join(parts)

print(search_act.name)
print(search_act.description)
print(search_act.args)
print(search_act.invoke({"query": "high-risk AI systems in employment"}))   # run the tool by hand

In [ ]:
# ▶ RUN · check: just run
done["1.2"] = "[A" in search_act.invoke({"query": "penalties for prohibited practices"})[:2]
print(done["1.2"])

**Reading the output.** The description is the docstring, word for word. The tool returns the passages as one text; this text is what the model reads in the tool message, and the labels let it name the article in its answer.

## 2. The Agent

With the search as a tool, the model decides how to search: it writes the query itself. As in Practice 1, `create_agent` builds the agent from the model, the tools, and the system prompt. The system prompt holds the grounding rules of a RAG answer, plus one rule about the tool and one about the number of searches.

### 2.1 ✍️ The system prompt and a first run

*Do:* rules 1 to 3 are written for you. Write rule 4 after `4.`: the agent may search at most 3 times and must then answer from what it has. Then run the cell. It builds the agent with the search tool, asks a question about a fine, and prints every message of the run.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
from langchain.agents import create_agent
from langchain.messages import HumanMessage

### FILL IN (START) ###
AGENT_SYSTEM = """You answer questions about the EU AI Act.
1. Use search_act to find passages. Answer only from the passages you retrieved; do not add facts from your own knowledge.
2. After each fact, name its article or annex, for example (Article 99(5)).
3. If the passages do not contain the answer, say so and name what is missing.
4. """
### FILL IN (END) ###

agent = create_agent(model, tools=[search_act], system_prompt=AGENT_SYSTEM)

FINE_QUESTION = "Under the EU AI Act, what is the maximum fine for supplying incorrect or misleading information to national authorities? Name the article."
result = agent.invoke({"messages": [HumanMessage(FINE_QUESTION)]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# ▶ RUN · check: just run
rule_lines = AGENT_SYSTEM.splitlines()
rule_4 = rule_lines[4].lower()
done["2.1"] = len(rule_lines) == 5 and ("3" in rule_4 or "three" in rule_4) and "search" in rule_4
print(done["2.1"])

**Reading the output.** Four messages: the question, an AI message with a tool call, the tool message with the passages, and the answer. The AI message with the tool call has no text, only the name `search_act` and the query that the model wrote. Compare the query with the question: the model kept only the words that matter for a search.

## 3. When to Search

Not every message needs a passage of the Act. In 2-step RAG the program would search anyway; here the model decides whether to call the tool, and it can answer at once.

### 3.1 ✍️ A message without a search

*Do:* write `NO_SEARCH_MESSAGE`: a message that the agent can answer without searching the Act, for example a greeting or a question about what it can do. Then run the cell. It sends your message to the agent of 2.1 and prints the messages of the run.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
### FILL IN (START) ###
NO_SEARCH_MESSAGE = ""                             # a message that the agent can answer without searching the Act
### FILL IN (END) ###

result_3 = agent.invoke({"messages": [HumanMessage(NO_SEARCH_MESSAGE)]})
for m in result_3["messages"]:
    m.pretty_print()

In [ ]:
# ▶ RUN · check: just run
calls_3 = 0
for m in result_3["messages"]:
    if m.type == "tool":
        calls_3 = calls_3 + 1
done["3.1"] = NO_SEARCH_MESSAGE != "" and calls_3 == 0
print(done["3.1"])

**Reading the output.** Two messages: the greeting and the answer. The agent did not call `search_act`, because the greeting needs no passage of the Act.

## 4. Searching Again

A first search can return passages that answer only part of the question. The model reads the result, decides whether it is enough, and if not, searches again, with a query that it writes from what it has just read. No rule in the prompt asks for this; the agent decides it.

### 4.1 A result that is not enough

*Do:* run the cell. It asks for the fine for a chatbot that does not tell its users that they are talking to an AI system, and prints every message of the run. Read the passages of each tool message and the query that follows them.

In [ ]:
# ▶ RUN · example: read each line, then run
CHATBOT_QUESTION = "What is the maximum fine for a company whose chatbot does not tell users that they are talking to an AI system?"
result_4 = agent.invoke({"messages": [HumanMessage(CHATBOT_QUESTION)]})
for m in result_4["messages"]:
    m.pretty_print()

**Reading the output.** In our runs the agent searched two or three times. The first search found Article 50, the duty to tell users that they are interacting with an AI system, but not the fine for breaking it. After reading this, the model searched again for the fine, found Article 99, and answered from both: up to EUR 15,000,000 or 3% of the turnover (Article 99(4)). Each new query was written after the model had read the result before it.

## 5. A Limit on the Loop

The model chooses how many times to search. Rule 4 asks for at most three searches, but a rule in the prompt is only a request. The program sets a hard limit with `recursion_limit`, a top-level key of the config of a call (Practice 1, 4.2): the maximum number of steps in one run, counting model calls and tool runs. A run that reaches the limit stops with `GraphRecursionError`.

### 5.1 A run that reaches the limit

*Do:* run the cell. It asks the question of 2.1 with a limit of 2 steps. A run with one search needs three: the model call that requests the search, the tool run, and the model call that writes the answer.

In [ ]:
# ▶ RUN · example: read each line, then run
from langgraph.errors import GraphRecursionError

TIGHT_LIMIT = {"recursion_limit": 2}             # at most 2 steps in one run
try:
    result = agent.invoke({"messages": [HumanMessage(FINE_QUESTION)]}, config=TIGHT_LIMIT)
    print("answered within 2 steps:", result["messages"][-1].content)
except GraphRecursionError:
    print("stopped by recursion_limit before the answer")

**Reading the output.** In our runs the model requested a search, so the run reached the limit before the answer and stopped: the stop came from the program, not from the prompt. In a real program the limit is set high enough for normal runs, so that it stops only a run that keeps requesting tools.

## 6. Memory

Each `invoke` so far started from an empty conversation. To continue a conversation, the agent gets a checkpointer and each call gets a thread id, as in Practice 1 (4.2). In a RAG agent the saved messages include the passages that the agent retrieved, so a follow-up question can use them.

### 6.1 ✍️ A follow-up question in one thread

*Do:* write `MY_FOLLOWUP`: a question that refers to the first answer without repeating it, for example about an exception, a deadline, or a duty that applies to "them". It must end with a question mark and must not contain the word "employment". The cell asks the first question in thread `"lesson-1"`, then your follow-up in the same thread, and prints the AI messages of the thread: the tool calls and the answers. The tool messages hold long passages, so the cell leaves them out.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
from langgraph.checkpoint.memory import InMemorySaver

agent_mem = create_agent(model, tools=[search_act], system_prompt=AGENT_SYSTEM, checkpointer=InMemorySaver())
thread_1 = {"configurable": {"thread_id": "lesson-1"}}

FIRST_QUESTION = "Which AI systems in the area of employment are high-risk?"

### FILL IN (START) ###
MY_FOLLOWUP = ""                                   # a follow-up that refers to the first answer, for example about exceptions or a deadline
### FILL IN (END) ###

## First question: the thread starts
result_first = agent_mem.invoke({"messages": [HumanMessage(FIRST_QUESTION)]}, config=thread_1)

## Follow-up: only the new question, in the same thread
result_follow = agent_mem.invoke({"messages": [HumanMessage(MY_FOLLOWUP)]}, config=thread_1)
for m in result_follow["messages"]:                # the thread: both questions with their tool calls
    if m.type == "ai":
        m.pretty_print()

In [ ]:
# ▶ RUN · check: just run
carried = len(result_follow["messages"]) > len(result_first["messages"])   # the follow-up run read the first run's messages
done["6.1"] = "?" in MY_FOLLOWUP and "employment" not in MY_FOLLOWUP.lower() and carried
print(done["6.1"])

**Reading the output.** The list holds both runs, because the thread kept the first one: the tool calls and the answer of the first question, then those of the follow-up. The follow-up says only "them", and the agent read it as the systems of the first answer; it searched again for the exception or answered from the passages it had already read. Every message stays in the thread, and the model reads all of them at each call, so a long conversation makes each call longer.

## Completion

*Do:* run the last cell after all the others. Each line is PASS or FAIL; LAB COMPLETE means that all checks passed. Then download the notebook with its outputs and upload it to the LMS.

In [ ]:
# ▶ RUN · check: just run
checks = {
    "name and student ID filled in (0)":                   STUDENT_NAME != "" and STUDENT_ID != "",
    "the search tool returns labelled passages (1.2)":     done["1.2"],
    "rule 4 limits the searches (2.1)":                    done["2.1"],
    "your message is answered without a search (3.1)":     done["3.1"],
    "your follow-up question in the thread (6.1)":         done["6.1"],
}
passed = 0
for name in checks:
    if checks[name]:
        print("PASS", name)
        passed = passed + 1
    else:
        print("FAIL", name)
print("submitted by:", STUDENT_NAME, STUDENT_ID)
if passed == len(checks):
    print("LAB COMPLETE")
else:
    print("NOT COMPLETE YET")